In [19]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import json
import numpy as np
import pandas as pd
from gensim.models import Word2Vec, FastText

In [3]:
DATASET_PATH = "/content/knowledge_base_improved 1.csv"
df = pd.read_csv(DATASET_PATH)

print("Original shape:", df.shape)
df.head()

Original shape: (50000, 5)


,document_id,category,title,content,keywords
0,KB000001,Authentication,Unable to Access Your Login Password Step by Step,If you can no longer use your sign-in password...,"password recovery, reset password, identity ve..."
1,KB000002,Authentication,Login Troubleshooting the Easy Way,Problems with your log in are often caused by ...,"login not working, login error, technical issu..."
2,KB000003,Authentication,Setting Up Extra Login Step,You can set up your two-step verification your...,"set up two-factor authentication, enable two-f..."
3,KB000004,Authentication,Verification Code Troubleshooting Explained,Most issues with your login code clear up afte...,"verification code not working, verification co..."
4,KB000005,Authentication,How to Set Up Your Authentication App the Easy...,Setting up your authentication app takes only ...,"set up authenticator app, enable authenticator..."


In [9]:
df.columns = [c.strip().lower() for c in df.columns]

required_cols = ["document_id", "category", "title", "content", "keywords"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(
        f"Missing expected column(s): {missing}. "
        f"Your actual columns are: {df.columns.tolist()}. "
        f"Update 'required_cols' or rename your CSV columns to match."
    )


In [10]:
def clean_and_tokenize(text):
    if pd.isna(text):
        return []
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)   # strip punctuation/special chars
    text = re.sub(r"\s+", " ", text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return tokens

In [11]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("punkt_tab")

STOPWORDS = set(stopwords.words("english"))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [13]:
df["title_tokens"] = df["title"].apply(clean_and_tokenize)
df["content_tokens"] = df["content"].apply(clean_and_tokenize)
df["keyword_tokens"] = df["keywords"].apply(clean_and_tokenize)

# Combined token list — this is what gets fed to Word2Vec/FastText training
# and also what gets embedded to build each document's vector representation.
df["combined_tokens"] = df["title_tokens"] + df["content_tokens"] +  df["keyword_tokens"]

# Keep a plain-text version too (useful later for the TF-IDF baseline,
# which expects raw/cleaned strings rather than token lists)
df["combined_text_clean"] = df["combined_tokens"].apply(lambda toks: " ".join(toks))

In [16]:
df.to_csv("cleaned_dataset.csv", index=False)

# Token lists only, saved separately since Word2Vec/FastText training
# (gensim) expects a list of token lists as input
import json
with open("training_tokens.json", "w") as f:
    json.dump(df["combined_tokens"].tolist(), f)

print("\nStep 4 complete.")
print(" - cleaned_dataset.csv   -> full cleaned data, for baseline TF-IDF & inspection")
print(" - training_tokens.json  -> tokenized corpus, for Word2Vec/FastText training in Step 5")
print(df[["document_id", "combined_tokens"]].head())


Step 4 complete.
 - cleaned_dataset.csv   -> full cleaned data, for baseline TF-IDF & inspection
 - training_tokens.json  -> tokenized corpus, for Word2Vec/FastText training in Step 5
  document_id                                    combined_tokens
0    KB000001  [unable, access, login, password, step, step, ...
1    KB000002  [login, troubleshooting, easy, way, problems, ...
2    KB000003  [setting, extra, login, step, set, two, step, ...
3    KB000004  [verification, code, troubleshooting, explaine...
4    KB000005  [set, authentication, app, easy, way, setting,...


In [20]:
with open("training_tokens.json", "r") as f:
    sentences = json.load(f)  # list of token lists

df = pd.read_csv("cleaned_dataset.csv")

print(f"Loaded {len(sentences)} tokenized documents.")

Loaded 50000 tokenized documents.


In [21]:
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    sg=1,
    workers=4,
    epochs=20,
)
w2v_model.save("word2vec.model")
print("Word2Vec training complete.")

Word2Vec training complete.


In [24]:
import json
import numpy as np
import pandas as pd
from gensim.models import FastText
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
STOPWORDS = set(stopwords.words("english"))

In [31]:
# ---------------------------------------------------------
# 1. Load everything from previous steps
# ---------------------------------------------------------
df = pd.read_csv("cleaned_dataset.csv")
df["combined_tokens"] = df["combined_tokens"].apply(eval)

ft_model = FastText.load("fasttext.model")
doc_vectors_ft = np.load("doc_vectors_fasttext.npy")

# ---------------------------------------------------------
# 2. Query preprocessing (same cleaning as training data, per 4.1)
# ---------------------------------------------------------
def clean_and_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 1]
    return tokens

def query_vector(query, model):
    tokens = clean_and_tokenize(query)
    vectors = []
    for token in tokens:
        try:
            vectors.append(model.wv[token])  # FastText handles unseen words too
        except KeyError:
            continue
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

In [42]:
def word2vec_search(query, top_n=5):
    q_vec = query_vector(query, w2v_model).reshape(1, -1)
    sims = cosine_similarity(q_vec, doc_vectors_w2v).flatten()
    ranked_idx = np.argsort(sims)[::-1][:top_n]
    results = df.iloc[ranked_idx].copy()
    results["similarity_score"] = sims[ranked_idx]
    return results[["document_id", "title", "category", "similarity_score"]]

In [63]:

    sample_query = input()  # replace with any test query    top_n = 5

    print(f"\nQuery: '{sample_query}'  (top {top_n} results)\n")

    print("=== Semantic Search (word2voc + cosine similarity) ===")
    result = word2vec_search(sample_query, top_n)
    result

password recovery

Query: 'password recovery'  (top 5 results)

=== Semantic Search (word2voc + cosine similarity) ===


,document_id,title,category,similarity_score
1311,KB001312,How to Reset Your Recovery Codes the Easy Way,Authentication,0.772062
448,KB000449,Fix Password Access Problems on the Web,Authentication,0.771965
1680,KB001681,Recover a Lost Account Password in Minutes,Authentication,0.771850
2912,KB002913,Reset Your Login Password Quick Guide,Authentication,0.771572
1536,KB001537,Regain Access to Your Sign-In Password,Authentication,0.769900


In [68]:
def fasttext_search(query, top_n=5):
    q_vec = query_vector(query, ft_model).reshape(1, -1)
    sims = cosine_similarity(q_vec, doc_vectors_ft).flatten()
    ranked_idx = np.argsort(sims)[::-1][:top_n]
    results = df.iloc[ranked_idx].copy()
    results["similarity_score"] = sims[ranked_idx]
    return results[["document_id", "title", "category", "similarity_score"]]

In [71]:
    sample_query = input()  # replace with any test query    top_n = 5

    print(f"\nQuery: '{sample_query}'  (top {top_n} results)\n")

    print("=== Semantic Search (fasttext_search + cosine similarity) ===")
    result = fasttext_search(sample_query, top_n)
    result

Unable to Access Your Login Password Step by Step

Query: 'Unable to Access Your Login Password Step by Step'  (top 5 results)

=== Semantic Search (fasttext_search + cosine similarity) ===


,document_id,title,category,similarity_score
3056,KB003057,What to Do If You Lose Your Login Password for...,Authentication,0.760760
992,KB000993,Resetting a Forgotten Sign-In Password for Bus...,Authentication,0.758078
2160,KB002161,Steps to Recover Your Password on Mobile,Authentication,0.751276
2240,KB002241,Unable to Access Your Account Password on Android,Authentication,0.750458
35566,KB035567,Forgot Your Password? on the Web,Authentication,0.746080


In [72]:
def tfidf_search(query, top_n=5):
    query_clean = " ".join(clean_and_tokenize(query))
    q_vec = tfidf_vectorizer.transform([query_clean])
    sims = cosine_similarity(q_vec, tfidf_matrix).flatten()
    ranked_idx = np.argsort(sims)[::-1][:top_n]
    results = df.iloc[ranked_idx].copy()
    results["similarity_score"] = sims[ranked_idx]
    return results[["document_id", "title", "category", "similarity_score"]]

In [74]:
    sample_query = input()  # replace with any test query    top_n = 5

    print(f"\nQuery: '{sample_query}'  (top {top_n} results)\n")

    print("=== Semantic Search (tfidf_search + cosine similarity) ===")
    result = tfidf_search(sample_query, top_n)
    result


password

Query: 'password'  (top 5 results)

=== Semantic Search (tfidf_search + cosine similarity) ===


,document_id,title,category,similarity_score
2400,KB002401,Unable to Access Your Account Password in the App,Authentication,0.842961
240,KB000241,Recover a Lost Sign-In Password FAQ,Authentication,0.837613
2016,KB002017,Get Back Into Your Account for New Users,Authentication,0.821326
2848,KB002849,Recover Your Password in Minutes,Authentication,0.820833
2032,KB002033,Sign-In Password Recovery Guide on Android,Authentication,0.815290
